# GymRAVANA progression-readiness data preparation and EDA

This notebook examines **genuine trainer-labeled, pseudonymized behavioral data** exported from Laravel. It does not train a model and it must not be presented as proof that an AI system already works.

## Before running

From the Laravel project root, run:

```powershell
php artisan gymravana:export-readiness-data
```

The generated CSV and metadata files remain local and are ignored by Git.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json

import pandas as pd

RANDOM_STATE = 42
working_directory = Path.cwd().resolve()
if working_directory.name == 'notebooks':
    PROJECT_ROOT = working_directory.parent.parent
elif (working_directory / 'ai').is_dir():
    PROJECT_ROOT = working_directory
else:
    raise RuntimeError('Open this notebook from the GymRAVANA project or ai/notebooks directory.')

DATASET_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.csv'
METADATA_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.metadata.json'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH}')

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError('Dataset not found. Run: php artisan gymravana:export-readiness-data')
if not METADATA_PATH.exists():
    raise FileNotFoundError('Dataset metadata not found. Export the dataset again before analysis.')

df = pd.read_csv(DATASET_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
actual_hash = sha256(DATASET_PATH.read_bytes()).hexdigest()
assert metadata.get('schema_version') == 1, 'Unsupported readiness dataset schema. Export it again with the current Laravel application.'
assert metadata.get('dataset_sha256') == actual_hash, 'CSV fingerprint does not match its metadata. Do not analyze an altered or mismatched export.'
assert metadata.get('row_count') == len(df), 'Metadata row count does not match the CSV.'
assert metadata.get('columns') == list(df.columns), 'Metadata columns do not match the CSV header.'
assert metadata.get('target') == 'ready_for_progression', 'Unexpected target in dataset metadata.'
print(f'Rows: {len(df)} | Columns: {len(df.columns)}')
print(json.dumps(metadata, indent=2))
display(df.head())

## Schema and privacy validation

Identifiers are pseudonymous grouping keys only. They must never be included as model inputs. Sensitive and free-text fields are forbidden.

In [ ]:
EXPECTED_COLUMNS = [
    'member_key', 'observation_month', 'label_recorded_at',
    'workout_completions', 'wellness_completions',
    'trainer_sessions_scheduled', 'trainer_sessions_completed',
    'attendance_rate', 'cancelled_or_declined_sessions',
    'active_days', 'consistency_rate', 'activity_points',
    'previous_goal_completion', 'previous_rating', 'previous_assessment',
    'workout_change', 'consistency_change', 'ready_for_progression',
]
FORBIDDEN_COLUMNS = {
    'user_id', 'trainer_profile_id', 'name', 'email', 'phone',
    'weight_kg', 'height_cm', 'waist_cm', 'chest_cm',
    'trainer_notes', 'readiness_rationale', 'therapy_request', 'diagnosis',
}

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
forbidden_columns = sorted(set(df.columns) & FORBIDDEN_COLUMNS)
print({'missing': missing_columns, 'unexpected': unexpected_columns, 'forbidden': forbidden_columns})
assert not missing_columns, f'Missing required columns: {missing_columns}'
assert not forbidden_columns, f'Sensitive or leakage-prone columns found: {forbidden_columns}'

## Missing values, duplicates and class distribution

In [ ]:
duplicate_key = ['member_key', 'observation_month', 'label_recorded_at']
duplicate_count = int(df.duplicated(subset=duplicate_key).sum()) if not df.empty else 0
conflicting_member_months = int((df.groupby(['member_key', 'observation_month'])['ready_for_progression'].nunique() > 1).sum()) if not df.empty else 0
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame('missing_count')
missing_summary['missing_percent'] = (missing_summary['missing_count'] / max(1, len(df)) * 100).round(2)
print(f'Duplicate observation keys: {duplicate_count}')
print(f'Conflicting member-month label groups: {conflicting_member_months}')
assert conflicting_member_months == 0, 'Contradictory readiness labels exist for the same member and observation month. Investigate them before analysis.'
display(missing_summary)

if df.empty:
    print('No genuine readiness labels exist yet. EDA can validate the schema, but model training is not allowed.')
else:
    display(df['ready_for_progression'].value_counts(dropna=False).rename_axis('label').to_frame('count'))

In [ ]:
NUMERIC_FEATURES = [
    'workout_completions', 'wellness_completions',
    'trainer_sessions_scheduled', 'trainer_sessions_completed',
    'attendance_rate', 'cancelled_or_declined_sessions',
    'active_days', 'consistency_rate', 'activity_points',
    'previous_goal_completion', 'previous_rating',
    'workout_change', 'consistency_change',
]
CATEGORICAL_FEATURES = ['previous_assessment']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'ready_for_progression'
NON_FEATURE_COLUMNS = {'member_key', 'observation_month', 'label_recorded_at', TARGET}

assert not (set(MODEL_FEATURES) & NON_FEATURE_COLUMNS)
if not df.empty:
    display(df[NUMERIC_FEATURES].describe().T)
else:
    print('Descriptive statistics will appear after genuine labeled rows are collected.')

## Leakage checks and split readiness

The current review's rating, assessment, goal completion, notes and rationale are intentionally absent because they may be recorded with the target label. Previous-month review values are lagged features. Member groups must not appear in both train and test data.

In [ ]:
class_counts = df[TARGET].value_counts() if not df.empty else pd.Series(dtype='int64')
member_group_count = int(df['member_key'].nunique()) if not df.empty else 0
has_both_classes = set(class_counts.index.tolist()) == {0, 1}
minimum_class_count = int(class_counts.min()) if has_both_classes else 0

# These are technical minimums for attempting a grouped split, not proof of adequate ML sample size.
MIN_MEMBER_GROUPS_FOR_SPLIT = 5
MIN_ROWS_PER_CLASS_FOR_SPLIT = 5
ready_for_split = (
    has_both_classes
    and member_group_count >= MIN_MEMBER_GROUPS_FOR_SPLIT
    and minimum_class_count >= MIN_ROWS_PER_CLASS_FOR_SPLIT
)

split_readiness = {
    'rows': len(df),
    'member_groups': member_group_count,
    'class_counts': class_counts.to_dict(),
    'has_both_classes': has_both_classes,
    'ready_for_grouped_split': ready_for_split,
}
print(json.dumps(split_readiness, indent=2))

In [ ]:
if ready_for_split:
    from sklearn.model_selection import GroupShuffleSplit
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_index, test_index = next(splitter.split(df[MODEL_FEATURES], df[TARGET], groups=df['member_key']))
    train_df = df.iloc[train_index].copy()
    test_df = df.iloc[test_index].copy()
    overlap = set(train_df['member_key']) & set(test_df['member_key'])
    assert not overlap, 'Member leakage detected between training and test groups.'
    print({'train_rows': len(train_df), 'test_rows': len(test_df), 'member_overlap': len(overlap)})
    print('Inspect each split class distribution before saving or training.')
else:
    train_df = pd.DataFrame(columns=df.columns)
    test_df = pd.DataFrame(columns=df.columns)
    print('Grouped train/test preparation skipped: collect more genuine labels across both classes and multiple members.')

## Decision gate before Notebook 02

Do not begin model comparison merely because the technical split minimum passes. Review label quality, class balance, missingness, group counts and whether the sample represents real GymRAVANA members. Synthetic records can test code but cannot be described as genuine model evidence.